# LaLiga football match forecasting dataset

This notebook downloads the reproducible v0.2.0 match-level foundation, builds leakage-safe pre-match features, adds the available StatsBomb historical LaLiga lineups, events, players, and managers, evaluates a small baseline, and publishes the dataset. The bootstrap is pinned to a reviewed repository commit, and the publication cell reads `HF_TOKEN` from Colab Secrets without prompting for or printing the token. Keep execution outputs cleared before committing or sharing the notebook.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

os.chdir('/content')
repo_url = 'https://github.com/EF-Code/laliga-match-forecasting.git'
repo_revision = '9c6e10fbe7641590049a510555519190ab4689bb'
requirements_url = f'https://raw.githubusercontent.com/EF-Code/laliga-match-forecasting/{repo_revision}/requirements-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', requirements_url], check=True)

repo_dir = Path('/content/laliga-match-forecasting')
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(['git', 'clone', '-q', '--no-checkout', repo_url, str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '-q', '--detach', repo_revision], check=True)
os.chdir(repo_dir)

In [ ]:
# Read HF_TOKEN in the notebook kernel, then pass it in memory to the Hub uploader.
# The token is never prompted for, printed, or written to disk.
import importlib
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path
from io import StringIO
from google.colab import userdata

importlib.invalidate_caches()
sys.path.insert(0, '/content/laliga-match-forecasting/src')
import laliga_forecasting.build_dataset as builder

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError('Add a write-capable HF_TOKEN secret in Colab and enable notebook access')

build_log = StringIO()
try:
    with redirect_stdout(build_log), redirect_stderr(build_log):
        output_dir = Path('/content/laliga-output')
        matches = builder.fetch_all_seasons()
        pre_match, observations, team_stats = builder.build_pre_match_dataset(matches)
        statsbomb_bundle = builder.build_statsbomb_bundle(output_dir / 'statsbomb-cache')
        paths = builder.write_outputs(output_dir, pre_match, observations, team_stats, statsbomb_bundle)
        builder.publish_to_hub(output_dir, pre_match, paths, token=hf_token)
    print('DATASET_PUBLISHED')
    print(f'MATCH_ROWS {len(matches)}')
    print(f'PRE_MATCH_ROWS {len(pre_match)}')
    print(f'TEAM_MATCH_ROWS {len(team_stats)}')
    for artifact_name, frame in statsbomb_bundle.items():
        print(f'{artifact_name.upper()}_ROWS {len(frame)}')
finally:
    del hf_token
    build_log.close()
    del build_log

In [ ]:
!PYTHONPATH=src python -m laliga_forecasting.train_baseline --input /content/laliga-output/pre_match_forecasting.parquet --output /content/laliga-output/baseline_metrics.json